# Combine datasets from PBMC Zheng et al.

In [4]:
from pathlib import Path
import tarfile

import anndata as ad
import pandas as pd
from scipy.io import mmread

INPUT = Path("/home/professor/Downloads/proj_data/dataset")  # Change to your downloads directory
OUTPUT = Path("/home/professor/Downloads/proj_data/pbmc_processed")
OUTPUT.mkdir(parents=True, exist_ok=True)

# Standardized population names, based on the downloaded dataset identities.
datasets = {
    "b_cells": "CD19+ B",
    "cd14_monocytes": "CD14+ Monocyte",
    "cd34": "CD34+",
    "cd4_t_helper": "CD4+ Helper T",
    "cd56_nk": "CD56+ NK",
    "cytotoxic_t": "CD8+ Cytotoxic T",
    "memory_t": "CD4+/CD45RO+ Memory T",
    "naive_cytotoxic": "CD8+/CD45RA+ Naive Cytotoxic T",
    "naive_t": "CD4+/CD45RA+/CD25- Naive T",
    "regulatory_t": "CD4+/CD25+ Regulatory T",
}

objects = []

for dataset, label in datasets.items():
    matches = list(
        INPUT.glob(f"{dataset}_filtered_gene_bc_matrices.tar.gz")
    )
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one archive for {dataset}; found {matches}"
        )

    with tarfile.open(matches[0], "r:gz") as archive:
        def find_member(filename):
            matches = [
                member for member in archive.getmembers()
                if member.isfile()
                and Path(member.name).name == filename
            ]
            if len(matches) != 1:
                raise ValueError(
                    f"{dataset}: expected one {filename}, "
                    f"found {len(matches)}"
                )
            return matches[0]

        with archive.extractfile(find_member("matrix.mtx")) as handle:
            # 10x files are genes × cells; AnnData uses cells × genes.
            counts = mmread(handle).T.tocsr()

        with archive.extractfile(find_member("genes.tsv")) as handle:
            genes = pd.read_csv(
                handle, sep="\t", header=None, dtype=str
            )

        with archive.extractfile(find_member("barcodes.tsv")) as handle:
            barcodes = pd.read_csv(
                handle, sep="\t", header=None, dtype=str
            )[0]

    assert counts.shape == (len(barcodes), len(genes))
    if genes[0].duplicated().any():
        raise ValueError(f"Duplicate gene IDs in {dataset}")

    obs = pd.DataFrame(
        {
            "barcode_original": barcodes.to_numpy(),
            "dataset": dataset,
            "cell_type_original": label,
            "label_source": "purified_population_identity",
        },
        index=pd.Index(
            [f"{dataset}:{barcode}" for barcode in barcodes],
            name="cell_id",
        ),
    )

    var = pd.DataFrame(
        {"gene_symbol": genes[1].to_numpy()},
        index=pd.Index(genes[0].to_numpy(), name="gene_id"),
    )

    obj = ad.AnnData(X=counts, obs=obs, var=var)

    # Stop rather than silently dropping or misaligning genes.
    if objects and not obj.var_names.equals(objects[0].var_names):
        raise ValueError(f"Gene IDs/order differ in {dataset}")

    objects.append(obj)
    print(f"{dataset}: {obj.n_obs:,} cells × {obj.n_vars:,} genes")

adata = ad.concat(objects, axis=0, join="inner", merge="same")
assert adata.obs_names.is_unique

# Remove genes absent from every cell; do not filter cells yet.
expressed = (adata.X.sum(axis=0).A1 > 0)
adata = adata[:, expressed].copy()

# Basic QC summaries, without changing the counts.
adata.obs["total_counts"] = adata.X.sum(axis=1).A1
adata.obs["n_genes_by_counts"] = adata.X.getnnz(axis=1)

adata.write_h5ad(
    OUTPUT / "zheng_purified_pbmc_counts.h5ad",
    compression="gzip",
)
adata.obs.to_csv(OUTPUT / "zheng_purified_pbmc_metadata.csv")

print("\nCombined shape:", adata.shape)
print(adata.obs["cell_type_original"].value_counts())

b_cells: 10,085 cells × 32,738 genes
cd14_monocytes: 2,612 cells × 32,738 genes
cd34: 9,232 cells × 32,738 genes
cd4_t_helper: 11,213 cells × 32,738 genes
cd56_nk: 8,385 cells × 32,738 genes
cytotoxic_t: 10,209 cells × 32,738 genes
memory_t: 10,224 cells × 32,738 genes
naive_cytotoxic: 11,953 cells × 32,738 genes
naive_t: 10,479 cells × 32,738 genes
regulatory_t: 10,263 cells × 32,738 genes

Combined shape: (94655, 21952)
cell_type_original
CD8+/CD45RA+ Naive Cytotoxic T    11953
CD4+ Helper T                     11213
CD4+/CD45RA+/CD25- Naive T        10479
CD4+/CD25+ Regulatory T           10263
CD4+/CD45RO+ Memory T             10224
CD8+ Cytotoxic T                  10209
CD19+ B                           10085
CD34+                              9232
CD56+ NK                           8385
CD14+ Monocyte                     2612
Name: count, dtype: int64
